# TME 05  
## Perceptron, SVM

## Imports

In [171]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mltools import plot_data, plot_frontiere, make_grid, gen_arti

### 1 - Perceptron et classe *Lineaire*

In [172]:
# Perceptron Loss
def perceptron_loss(w, x, y) : 
    y = y.reshape((y.shape[0], 1))
    agg_mult = x@w * y
    loss = np.where(agg_mult<0, -agg_mult, 0)
    return loss

In [173]:
# Perceptron Gradient
def perceptron_grad(w, x, y):
    # Compute loss 
    loss = perceptron_loss(w, x, y)
    print(loss.shape, w.shape, y.shape, x.shape)
    y = y.reshape((y.shape[0], 1))
    loss_grad = np.where(loss==0, 0, -x*y)
    print(loss_grad.shape)
    return loss_grad

In [174]:
class Lineaire(object):
    def __init__(self,loss=perceptron_loss,loss_g=perceptron_grad,max_iter=100,eps=0.01):
        self.max_iter, self.eps = max_iter,eps
        self.w = None
        self.loss,self.loss_g = loss,loss_g
        
    def fit(self,datax,datay):
        if self.w == None : 
            d = datax.shape[1]
            self.w = np.zeros((d, 1))
            
        for i in range(self.max_iter) :
            grad = self.loss_g(self.w, datax, datay)
            new_w = self.w-self.eps*grad

            if np.allclose(self.w, new_w, 1e-1) :
                return new_w # Convergence
            else :
                self.w = new_w # Update

        return new_w

    def predict(self,datax):
        agg = datax@self.w
        act = np.sign(agg)
        return act

    def score(self,datax,datay):
        loss = self.loss(self.w, datax, datay)
        n = loss.shape[0] # nb of points x
        score = np.sum(loss)/n # score in percentage

        return score

In [175]:
def load_usps(fn):
    with open(fn,"r") as f:
        f.readline()
        data = [[float(x) for x in l.split()] for l in f if len(l.split())>2]
    tmp=np.array(data)
    return tmp[:,1:],tmp[:,0].astype(int)

def get_usps(l,datax,datay):
    if type(l)!=list:
        resx = datax[datay==l,:]
        resy = datay[datay==l]
        return resx,resy
    tmp =   list(zip(*[get_usps(i,datax,datay) for i in l]))
    tmpx,tmpy = np.vstack(tmp[0]),np.hstack(tmp[1])
    return tmpx,tmpy

In [176]:
if __name__ =="__main__":
    uspsdatatrain = "../data/USPS_train.txt"
    uspsdatatest = "../data/USPS_test.txt"
    alltrainx,alltrainy = load_usps(uspsdatatrain)
    alltestx,alltesty = load_usps(uspsdatatest)
    neg = 5
    pos = 6
    datax,datay = get_usps([neg,pos],alltrainx,alltrainy)
    testx,testy = get_usps([neg,pos],alltestx,alltesty)

### Test *Lineaire*

In [177]:
# Create data
datax,datay = get_usps([neg,pos],alltrainx,alltrainy)

# Create instance of Lineaire
lin = Lineaire()

# Optimization
w = lin.fit(datax, datay)

# Prediction
y_hat = lin.predict(w, datax, datay)

print(y_hat)

# Compute accuracy
acc = lin.score(datax, datay)
print(acc)


(1220, 1) (256, 1) (1220,) (1220, 256)
(1220, 256)


ValueError: operands could not be broadcast together with shapes (256,1) (1220,256) 